In [1]:
import pandas as pd
import requests
from io import StringIO
import numpy as np

# Сбор данных

**kepid**, **kepoi_name**, **kepler_name** - id/имена космических объектов. Возможно, будут служить ключами для последующего соединения с данными о кривых блеска соответствующих звёзд

**koi_pdisposition** и **koi_score** - рещультаты предсказания модели nasa - полезно посмотреть, нельзя передавать в модель во избежание data leakage

**koi_disposition** - целевая переменная, нужно научиться предсказать

**Остальные параметры** - метрики для обучения модели

In [2]:
columns = [
    "kepid", "kepoi_name", "kepler_name",
    "koi_disposition", "koi_pdisposition", "koi_score",
    "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec",
    "koi_period", "koi_duration", "koi_depth", "koi_prad",
    "koi_teq", "koi_insol", "koi_model_snr",
    "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag",
]
query = f"select {','.join(columns)} from cumulative"

response = requests.get(
    "https://exoplanetarchive.ipac.caltech.edu/TAP/sync",
    params={"query": query, "format": "csv"}
)
response.raise_for_status()

koi = pd.read_csv(StringIO(response.text))
koi.to_csv("../data/koi_cumulative.csv", index=False)

# Структура скаченных данных

In [3]:
print(koi.shape)
print(koi["koi_disposition"].value_counts())
koi.head(10)

(9564, 21)
koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
CANDIDATE         1977
Name: count, dtype: int64


,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,...,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,...,2.95750,615.8,2.26,793.0,93.59,35.8,5455.0,4.467,0.927,15.347
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,CANDIDATE,0.969,0,0,0,0,...,4.50700,874.8,2.83,443.0,9.11,25.8,5455.0,4.467,0.927,15.347
2,10811496,K00753.01,NaN,CANDIDATE,CANDIDATE,0.000,0,0,0,0,...,1.78220,10829.0,14.60,638.0,39.30,76.3,5853.0,4.544,0.868,15.436
3,10848459,K00754.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,0,0,...,2.40641,8079.2,33.46,1395.0,891.96,505.6,5805.0,4.564,0.791,15.597
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,...,1.65450,603.3,2.75,1406.0,926.16,40.9,6031.0,4.438,1.046,15.509
5,10872983,K00756.01,Kepler-228 d,CONFIRMED,CANDIDATE,1.000,0,0,0,0,...,4.59450,1517.5,3.90,835.0,114.81,66.5,6046.0,4.486,0.972,15.714
6,10872983,K00756.02,Kepler-228 c,CONFIRMED,CANDIDATE,1.000,0,0,0,0,...,3.14020,686.0,2.77,1160.0,427.65,40.2,6046.0,4.486,0.972,15.714
7,10872983,K00756.03,Kepler-228 b,CONFIRMED,CANDIDATE,0.992,0,0,0,0,...,2.42900,226.5,1.59,1360.0,807.74,15.0,6046.0,4.486,0.972,15.714
8,6721123,K00114.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,1,0,...,5.02200,233.7,39.21,1342.0,767.22,47.7,6227.0,3.986,1.958,12.660
9,10910878,K00757.01,Kepler-229 c,CONFIRMED,CANDIDATE,1.000,0,0,0,0,...,3.53470,4914.3,5.76,600.0,30.75,161.9,5031.0,4.485,0.848,15.841


In [4]:
koi.describe()

,kepid,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag
count,9.564000e+03,8054.000000,9564.000000,9564.000000,9564.000000,9564.000000,9564.000000,9564.000000,9.201000e+03,9201.000000,9201.000000,9.243000e+03,9201.000000,9201.000000,9201.000000,9201.000000,9563.000000
mean,7.690628e+06,0.480829,0.208595,0.232748,0.197512,0.120033,75.671358,5.621606,2.379134e+04,102.891778,1085.385828,7.745737e+03,259.895001,5706.823280,4.310157,1.728712,14.264606
std,2.653459e+06,0.476928,4.767290,0.422605,0.398142,0.325018,1334.744046,6.471554,8.224268e+04,3077.639126,856.351161,1.592047e+05,795.806615,796.857947,0.432606,6.127185,1.385448
min,7.574500e+05,0.000000,0.000000,0.000000,0.000000,0.000000,0.241843,0.052000,0.000000e+00,0.080000,25.000000,0.000000e+00,0.000000,2661.000000,0.047000,0.109000,6.966000
25%,5.556034e+06,0.000000,0.000000,0.000000,0.000000,0.000000,2.733684,2.437750,1.599000e+02,1.400000,539.000000,2.015000e+01,12.000000,5310.000000,4.218000,0.829000,13.440000
50%,7.906892e+06,0.334000,0.000000,0.000000,0.000000,0.000000,9.752831,3.792600,4.211000e+02,2.390000,878.000000,1.416000e+02,23.000000,5767.000000,4.438000,1.000000,14.520000
75%,9.873066e+06,0.998000,0.000000,0.000000,0.000000,0.000000,40.715178,6.276500,1.473400e+03,14.930000,1379.000000,8.702900e+02,78.000000,6112.000000,4.543000,1.345000,15.322000
max,1.293514e+07,1.000000,465.000000,1.000000,1.000000,1.000000,129995.778400,138.540000,1.541400e+06,200346.000000,14667.000000,1.094755e+07,9054.700000,15896.000000,5.364000,229.908000,20.003000


In [5]:
koi.columns

Index(['kepid', 'kepoi_name', 'kepler_name', 'koi_disposition',
       'koi_pdisposition', 'koi_score', 'koi_fpflag_nt', 'koi_fpflag_ss',
       'koi_fpflag_co', 'koi_fpflag_ec', 'koi_period', 'koi_duration',
       'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_model_snr',
       'koi_steff', 'koi_slogg', 'koi_srad', 'koi_kepmag'],
      dtype='object')

In [6]:
koi[["koi_disposition", "koi_pdisposition", "koi_score", "kepoi_name", "kepid", "kepler_name"]].head()

,koi_disposition,koi_pdisposition,koi_score,kepoi_name,kepid,kepler_name
0,CONFIRMED,CANDIDATE,1.000,K00752.01,10797460,Kepler-227 b
1,CONFIRMED,CANDIDATE,0.969,K00752.02,10797460,Kepler-227 c
2,CANDIDATE,CANDIDATE,0.000,K00753.01,10811496,NaN
3,FALSE POSITIVE,FALSE POSITIVE,0.000,K00754.01,10848459,NaN
4,CONFIRMED,CANDIDATE,1.000,K00755.01,10854555,Kepler-664 b


# Просмотр пропусков

Пропуски в kepler_name можно не учитывать - это естественно, что их так много, т. к. имена дают только подтверждённым космическим объектам. Так как этот параметр не пойдёт в модель, можно оставить как есть

koi_score - аналогично: просто уберётся столбец до передачи данных модели, чтобы она не подсматривала результаты модели nasa

Остальные пропуски - 363 пропуска в почти что половине колонок. Если это всё одни и те же строки, можно спокойно удалить. Проверим это, убрав пропуски по одному из этих параметров и посмотреть isna заново: если пропусков больше нет - значит, гипотеза была верна

In [7]:
koi.isna().sum()

kepid                  0
kepoi_name             0
kepler_name         6816
koi_disposition        0
koi_pdisposition       0
koi_score           1510
koi_fpflag_nt          0
koi_fpflag_ss          0
koi_fpflag_co          0
koi_fpflag_ec          0
koi_period             0
koi_duration           0
koi_depth            363
koi_prad             363
koi_teq              363
koi_insol            321
koi_model_snr        363
koi_steff            363
koi_slogg            363
koi_srad             363
koi_kepmag             1
dtype: int64

Просто было интересно посмотреть

In [8]:
koi[koi["koi_score"].isna() == True]

,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,...,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag
24,2306756,K00113.01,NaN,FALSE POSITIVE,FALSE POSITIVE,NaN,0,1,0,0,...,6.8057,24926.6,41.50,297.0,1.84,994.0,5543.0,4.081,1.580,12.394
75,12459725,K00789.01,NaN,FALSE POSITIVE,FALSE POSITIVE,NaN,0,0,0,1,...,2.5050,459.9,1.88,711.0,60.35,11.2,5799.0,4.539,0.878,15.727
99,3832474,K00806.03,Kepler-30 b,CONFIRMED,CANDIDATE,NaN,0,0,0,0,...,4.8030,488.0,1.91,524.0,17.85,11.6,5485.0,4.556,0.867,15.403
123,5115978,K00823.01,NaN,CANDIDATE,CANDIDATE,NaN,0,0,0,0,...,1.8606,8098.0,72.77,1947.0,3400.48,89.6,6228.0,4.428,0.990,15.202
318,10460629,K01168.01,NaN,CANDIDATE,CANDIDATE,NaN,0,0,0,0,...,23.1440,858.1,3.87,242.0,0.81,39.2,6449.0,4.232,1.309,13.997
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9155,9025662,K07599.01,NaN,FALSE POSITIVE,FALSE POSITIVE,NaN,1,0,0,0,...,11.0300,1575.0,6140.44,274.0,1.34,11.2,6054.0,4.508,0.893,15.510
9162,5652983,K00371.02,NaN,FALSE POSITIVE,FALSE POSITIVE,NaN,0,1,1,0,...,6.3760,61.8,2.46,1310.0,695.41,17.9,5198.0,3.614,3.207,12.193
9163,9092897,K07600.01,NaN,FALSE POSITIVE,FALSE POSITIVE,NaN,1,0,0,0,...,2.1100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.940
9165,9292130,K07602.01,NaN,FALSE POSITIVE,FALSE POSITIVE,NaN,1,0,0,0,...,16.4500,132.4,2.51,384.0,5.13,13.0,6176.0,3.885,2.160,10.662


In [9]:
koi

,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,...,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,...,2.95750,615.8,2.26,793.0,93.59,35.8,5455.0,4.467,0.927,15.347
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,CANDIDATE,0.969,0,0,0,0,...,4.50700,874.8,2.83,443.0,9.11,25.8,5455.0,4.467,0.927,15.347
2,10811496,K00753.01,NaN,CANDIDATE,CANDIDATE,0.000,0,0,0,0,...,1.78220,10829.0,14.60,638.0,39.30,76.3,5853.0,4.544,0.868,15.436
3,10848459,K00754.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,0,0,...,2.40641,8079.2,33.46,1395.0,891.96,505.6,5805.0,4.564,0.791,15.597
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,...,1.65450,603.3,2.75,1406.0,926.16,40.9,6031.0,4.438,1.046,15.509
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9559,10090151,K07985.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,1,0,...,3.22210,1579.2,29.35,2088.0,4500.53,453.3,5638.0,4.529,0.903,14.082
9560,10128825,K07986.01,NaN,CANDIDATE,CANDIDATE,0.497,0,0,0,0,...,3.11400,48.5,0.72,1608.0,1585.81,10.6,6119.0,4.444,1.031,14.757
9561,10147276,K07987.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.021,0,0,1,0,...,0.86500,103.6,1.07,2218.0,5713.41,12.3,6173.0,4.447,1.041,15.385
9562,10155286,K07988.01,NaN,CANDIDATE,CANDIDATE,0.092,0,0,0,0,...,3.19900,639.1,19.30,557.0,22.68,14.0,4989.0,2.992,7.824,10.998


In [10]:
koi.drop(columns=["koi_score", "koi_pdisposition"], inplace=True)
koi.dropna(subset=["koi_depth", "koi_kepmag"], inplace=True)

Всё отлично, пропусков больше нет

In [11]:
koi.isna().sum()

kepid                 0
kepoi_name            0
kepler_name        6454
koi_disposition       0
koi_fpflag_nt         0
koi_fpflag_ss         0
koi_fpflag_co         0
koi_fpflag_ec         0
koi_period            0
koi_duration          0
koi_depth             0
koi_prad              0
koi_teq               0
koi_insol             0
koi_model_snr         0
koi_steff             0
koi_slogg             0
koi_srad              0
koi_kepmag            0
dtype: int64

Так как это данные о разных звёздах с официального сайта nasa, дубликатов быть не должно, но всё равно можно проверить

In [12]:
display(koi.shape)
koi.drop_duplicates(inplace=True)   # Только полные дубликаты
koi.shape

(9200, 19)

(9200, 19)

# Стандартизация данных

Делаем масштабирование только по числовым признакам

In [13]:
col_for_scaling = koi.drop(columns=["kepid", "kepoi_name", "kepler_name", 'koi_disposition', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec']).columns

In [14]:
col_for_scaling

Index(['koi_period', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq',
       'koi_insol', 'koi_model_snr', 'koi_steff', 'koi_slogg', 'koi_srad',
       'koi_kepmag'],
      dtype='object')

Смотрим, насколько существенные имеются выбросы в этих признаках, так как стандартизация чувствительна к сильным выбросам, а датасет основан на информации о разных звёздах с абсолютно разными показателями, которые могут отличаться кардинально.

Если skew будет показывать для признака число, значительно отличное от 1 - до стандартизации лучше применить логарифмирование

In [15]:
print(koi[col_for_scaling].skew().sort_values(ascending=False))

koi_period       94.668430
koi_prad         52.116120
koi_insol        49.852434
koi_srad         20.945969
koi_duration      5.872071
koi_model_snr     5.306784
koi_depth         5.260440
koi_teq           3.505657
koi_steff         0.749772
koi_kepmag       -0.809560
koi_slogg        -3.432862
dtype: float64


In [16]:
log_cols = ["koi_period", "koi_prad", "koi_insol", "koi_srad", "koi_duration", "koi_model_snr", "koi_depth", "koi_teq", "koi_slogg"]

In [17]:
for col in log_cols:
    if ((koi[col] >= 0).all()):
        koi[col] = np.log1p(koi[col])
        print(f"{col} прошёл логарифмизацию")

koi_period прошёл логарифмизацию
koi_prad прошёл логарифмизацию
koi_insol прошёл логарифмизацию
koi_srad прошёл логарифмизацию
koi_duration прошёл логарифмизацию
koi_model_snr прошёл логарифмизацию
koi_depth прошёл логарифмизацию
koi_teq прошёл логарифмизацию
koi_slogg прошёл логарифмизацию


In [18]:
koi

,kepid,kepoi_name,kepler_name,koi_disposition,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,0,0,0,0,2.350235,1.375613,6.424545,1.181727,6.677083,4.549552,3.605498,5455.0,1.698730,0.655964,15.347
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,0,0,0,0,4.014911,1.706020,6.775138,1.342865,6.095825,2.313525,3.288402,5455.0,1.698730,0.655964,15.347
2,10811496,K00753.01,NaN,CANDIDATE,0,0,0,0,3.039708,1.023242,9.290075,2.747271,6.459904,3.696351,4.347694,5853.0,1.712716,0.624868,15.436
3,10848459,K00754.01,NaN,FALSE POSITIVE,0,1,0,0,1.006845,1.225659,8.997172,3.539799,7.241366,6.794542,6.227722,5805.0,1.716317,0.582774,15.597
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,0,0,0,0,1.260048,0.976256,6.404071,1.321756,7.249215,6.832126,3.735286,6031.0,1.693411,0.715887,15.509
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9559,10090151,K07985.01,NaN,FALSE POSITIVE,0,1,1,0,0.423762,1.440333,7.365307,3.412797,7.644441,8.412173,6.118758,5638.0,1.710007,0.643432,14.082
9560,10128825,K07986.01,NaN,CANDIDATE,0,0,0,0,1.007903,1.414396,3.901973,0.542324,7.383368,7.369481,2.451005,6119.0,1.694514,0.708528,14.757
9561,10147276,K07987.01,NaN,FALSE POSITIVE,0,0,1,0,0.519628,0.623261,4.650144,0.727549,7.704812,8.650746,2.587764,6173.0,1.695065,0.713440,15.385
9562,10155286,K07988.01,NaN,CANDIDATE,0,0,0,0,5.812596,1.434846,6.461624,3.010621,6.324359,3.164631,2.708050,4989.0,1.384292,2.177475,10.998


In [19]:
for col in col_for_scaling:
    koi[col] = (koi[col] - koi[col].mean())/koi[col].std()

In [20]:
koi

,kepid,kepoi_name,kepler_name,koi_disposition,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,0,0,0,0,-0.177514,-0.469595,-0.094705,-0.477763,-0.108628,-0.153479,-0.074988,-0.315993,0.325533,-0.353040,0.777070
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,0,0,0,0,0.832140,0.066819,0.053967,-0.361913,-0.946123,-1.013427,-0.272397,-0.315993,0.325533,-0.353040,0.777070
2,10811496,K00753.01,NaN,CANDIDATE,0,0,0,0,0.240663,-1.041665,1.120446,0.647789,-0.421546,-0.481609,0.387069,0.183441,0.461198,-0.428091,0.841648
3,10848459,K00754.01,NaN,FALSE POSITIVE,0,1,0,0,-0.992303,-0.713043,0.996238,1.217580,0.704408,0.709916,1.557487,0.123208,0.496127,-0.529688,0.958468
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,0,0,0,0,-0.838731,-1.117946,-0.103387,-0.377089,0.715717,0.724371,0.005812,0.406807,0.273943,-0.208415,0.894616
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9559,10090151,K07985.01,NaN,FALSE POSITIVE,0,1,1,0,-1.345952,-0.364522,0.304233,1.126271,1.285170,1.332037,1.489651,-0.086354,0.434918,-0.383288,-0.140802
9560,10128825,K07986.01,NaN,CANDIDATE,0,0,0,0,-0.991661,-0.406631,-1.164422,-0.937464,0.909009,0.931031,-0.793721,0.517235,0.284639,-0.226174,0.348972
9561,10147276,K07987.01,NaN,FALSE POSITIVE,0,0,1,0,-1.287808,-1.691031,-0.847154,-0.804297,1.372154,1.423789,-0.708582,0.584997,0.289983,-0.214320,0.804643
9562,10155286,K07988.01,NaN,CANDIDATE,0,0,0,0,1.922466,-0.373429,-0.078981,0.837126,-0.616844,-0.686102,-0.633697,-0.900759,-2.724468,3.319194,-2.378525


In [21]:
koi["koi_disposition"].value_counts()

koi_disposition
FALSE POSITIVE    4581
CONFIRMED         2746
CANDIDATE         1873
Name: count, dtype: int64

Разделяем данные: так как значение CANDIDATE метрики koi_disposition, по сути, несёт в себе только отсутствие решения модели nasa, это достаточно сильно может запутать будущую модель, заставив её не определять, имеет ли звезда экзопланету, а просто подгонять значения под выводф модели nasa. Стоит убрать строки с этим значением, оставив для модели задачу бинарной классификации (CONFIRMED/FALSE POSITIVE), но строки с CANDIDATE оставим отдельно для возможного будущего изучения

In [22]:
koi_candidates = koi[koi["koi_disposition"] == "CANDIDATE"].drop(columns=["kepid", "kepoi_name", "kepler_name"]).copy()
koi_binary = koi[koi["koi_disposition"] != "CANDIDATE"].drop(columns=["kepid", "kepoi_name", "kepler_name"]).copy()
keys = koi[["kepid", "kepoi_name", "kepler_name"]].copy()  # сохраним все метрики-идентификаторы тут

In [23]:
display(koi_candidates["koi_disposition"].value_counts())
display(koi_binary["koi_disposition"].value_counts())
keys.head()

koi_disposition
CANDIDATE    1873
Name: count, dtype: int64

koi_disposition
FALSE POSITIVE    4581
CONFIRMED         2746
Name: count, dtype: int64

,kepid,kepoi_name,kepler_name
0,10797460,K00752.01,Kepler-227 b
1,10797460,K00752.02,Kepler-227 c
2,10811496,K00753.01,NaN
3,10848459,K00754.01,NaN
4,10854555,K00755.01,Kepler-664 b
